# Optimizing Insurance Pricing with Response Surface Methodology (PROC RSREG)

## Executive Summary

An auto insurer runs a controlled two-factor experiment to find the combination of **premium discount** and **engagement spend** that **maximizes net profit per policy**. `PROC RSREG` fits a quadratic response surface to the 52-run central composite design, then runs the **canonical analysis** to classify the surface and a **ridge analysis** to trace the path of steepest improvement out of the design center.

The fitted surface explains the response almost completely (**R-square = 0.974**). The canonical analysis returns a **Maximum** — both eigenvalues of the quadratic form are negative (**-1.94** and **-0.058**) — at a stationary point of **8.87% discount** and **$34.55 engagement spend**, with a **predicted net profit of $143.33 per policy**. The ridge of maximum response climbs monotonically from the design center ($139.86) toward that peak, giving the pricing team an ordered sequence of settings to confirm in a pilot.

## Data Sources

| Dataset | Rows | Description |
|---|---|---|
| `pricing_doe` | 52 | Synthetic central composite design (CCD) for an insurance pricing experiment: 4 factorial corners, 4 axial/star points, and a center point, each replicated 4 times. |

| Variable | Type | Description |
|---|---|---|
| `Discount` | Num | Premium discount offered to the policyholder (%), centered at 8% (range ~5–11%). |
| `Engagement` | Num | Engagement / outreach spend per policy ($), centered at \$30 (range ~\$19–\$41). |
| `Profit` | Num | Observed net profit per policy ($) — a quadratic function of the two factors plus Gaussian noise, with a true interior maximum. |

# Optimizing Insurance Pricing with Response Surface Methodology

A personal-auto carrier wants to set two retention levers at the same time: how aggressive a **premium discount** to offer at renewal, and how much **engagement spend** (telematics nudges, app outreach, agent follow-up) to invest per policy. Too small a discount loses customers; too large erodes margin. Engagement spend behaves similarly — there is a sweet spot.

Because the profit response curves in *both* factors and the two levers interact, a designed experiment plus a **quadratic response surface** is the right tool. We use a **central composite design (CCD)** and fit it with `PROC RSREG`, which fits the full second-order model, performs a **canonical analysis** to classify the surface (maximum, minimum, or saddle), and runs a **ridge analysis** to trace the path of steepest improvement toward the optimum.

## Step 1 — Generate the experimental data

We simulate a rotatable central composite design in two coded factors: four factorial corners at (±1, ±1), four axial ("star") points at (±√2, 0) and (0, ±√2), and a center point at (0, 0). The 13-point block is replicated four times (52 runs) so the model is estimated from repeated observations at each design location, giving a stable fit.

The coded factors are decoded to business units: discount centered at 8% (±2% per coded unit) and engagement spend centered at \$30 (±\$8 per coded unit). The true profit surface has an interior maximum, with curvature and an interaction term, plus realistic measurement noise.

In [1]:
data pricing_doe;
   call streaminit(20260531);
   label Discount   = "Premium Discount (%)"
         Engagement = "Engagement Spend ($/policy)"
         Profit     = "Net Profit per Policy ($)";

   /* Coded central composite design:
      4 factorial corners + 4 axial points + 1 center point */
   array px{13} _temporary_
      (-1 -1  1  1 -1.414  1.414  0      0      0 0 0 0 0);
   array py{13} _temporary_
      (-1  1 -1  1  0      0     -1.414  1.414  0 0 0 0 0);

   do rep = 1 to 4;
      do pt = 1 to 13;
         x1 = px{pt};
         x2 = py{pt};
         Discount   = 8 + 2*x1;    /* 8% center, +/-2% per coded unit  */
         Engagement = 30 + 8*x2;   /* $30 center, +/-$8 per coded unit */
         /* True quadratic surface with an interior maximum + noise */
         Profit = 140 + 9*x1 + 6*x2 - 7*x1*x1 - 5*x2*x2 - 3*x1*x2
                  + rand("normal", 0, 2.5);
         output;
      end;
   end;
   keep Discount Engagement Profit;
run;

NOTE: DATA pricing_doe


NOTE: Wrote pricing_doe (52 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Step 2 — Sanity-check the design

Before modeling, confirm the design spans the intended ranges and is balanced around the center. The factor means should sit at the design centers (8% and \$30), and the response should vary meaningfully across the runs.

In [2]:
proc means data=pricing_doe n mean min max maxdec=2;
   var Discount Engagement Profit;
   title "Design Summary: Insurance Pricing Experiment";
run;

                                      Design Summary: Insurance Pricing Experiment                                      

                                                  The MEANS Procedure

 Variable                           N           Mean     Minimum     Maximum
 ---------------------------------------------------------------------------
 Premium Discount (%)              52           8.00        5.17       10.83
 Engagement Spend ($/policy)       52          30.00       18.69       41.31
 Net Profit per Policy ($)         52         132.62      107.17      142.94
 ---------------------------------------------------------------------------



NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Step 3 — Fit the response surface and find the optimum

`PROC RSREG` fits the full quadratic model `Profit = f(Discount, Engagement)` including the linear, pure-quadratic, and cross-product terms. We add `RIDGE MAX` so the procedure computes the **ridge of maximum response** — the path of locally optimal factor settings as we move outward from the design center.

Read three blocks of the output:

- **Estimated Regression Coefficients** — the fitted second-order model. Negative pure-quadratic coefficients are what create a peak rather than a monotone trend.
- **Stationary Point + Eigenvalues of the Quadratic Matrix** (the canonical analysis) — the critical factor values, the predicted response there, and whether the surface is a maximum, minimum, or saddle. Two negative eigenvalues confirm a true maximum.
- **Ridge Analysis (MAX Direction)** — the optimal trajectory and predicted response as the coded radius grows from 0 (the center) outward.

In [3]:
proc rsreg data=pricing_doe;
   model Profit = Discount Engagement;
   ridge max;
   title "Response Surface for Net Profit per Policy";
run;

                                      Design Summary: Insurance Pricing Experiment                                      


                       The RSREG Procedure                       

Response Variable: PROFIT: Net Profit per Policy ($)
Number of Observations: 52
Mean Response: 132.6228
R-Square = 0.974139
Coefficient of Variation: 1.3308
Root MSE: 1.7650

Estimated Regression Coefficients
--------------------------------------------------------------------
Parameter                  Estimate    StdErr  t Value  Pr > |t|  DF
Intercept                -139.95794   9.97965  -14.024     0.0000   1
Discount                   40.73680   1.58140   25.760     0.0000   1
Engagement                  5.94609   0.38553   15.423     0.0000   1
Discount*Discount          -1.93981   0.08367  -23.185     0.0000   1
Engagement*Engagement      -0.06251   0.00523  -11.953     0.0000   1
Discount*Engagement        -0.18357   0.02758   -6.656     0.0000   1

Regression                       DF  R-Squ

NOTE: PROC RSREG data=pricing_doe



## Step 4 — Interpreting the results

**Model fit.** The quadratic model explains **97.4%** of the variation in net profit (R-square = 0.974, root MSE = 1.77 on a mean response of $132.62). Every term is strongly significant: both linear effects are positive (Discount 40.74, Engagement 5.95) while both pure-quadratic coefficients are **negative** (Discount² = -1.94, Engagement² = -0.063) — the signature of a response that rises, peaks, then falls. The negative `Discount*Engagement` cross-product (-0.18) means the two levers are partial substitutes: heavy discounting trims the marginal payoff from extra engagement spend, and vice versa.

**Canonical analysis.** The stationary point is classified as a **Maximum**, with both eigenvalues of the quadratic matrix negative (**-1.94** and **-0.058**) — the surface is concave-down in every direction, so the critical point is a genuine profit peak rather than a saddle or minimum. The critical values translate directly into a recommended operating point: a premium discount of **8.87%** paired with engagement spend of **$34.55 per policy**, where the model predicts net profit of **$143.33 per policy** — about $11 above the average run in the experiment.

**Ridge analysis.** The *Ridge Analysis (MAX Direction)* table traces the optimal path outward from the design center (8%, $30, predicted **$139.86**). Estimated net profit climbs monotonically along the ridge — $140.33, $140.75, … up to **$142.59** at coded radius 1.0 (discount 9.0%, engagement $31.0) — heading toward the stationary point. This gives the pricing team a concrete, ordered sequence of factor settings to test in a confirmatory pilot rather than a single leap to the optimum.

**Business takeaway.** Rather than tuning discount and engagement spend one at a time — which would miss their interaction — the response surface pinpoints the joint setting that maximizes net profit per policy and shows how quickly profit falls off as the levers drift from the optimum. The carrier can roll out the recommended discount/engagement combination in a controlled pilot, using the ridge as a roadmap if real-world constraints force a move away from the exact peak.